In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/application_train.csv")
df.shape

(307511, 122)

In [2]:
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), str(16)
memory usage: 286.2 MB


In [4]:
df["TARGET"].value_counts()

TARGET
0    282686
1     24825
Name: count, dtype: int64

In [5]:
df["TARGET"].value_counts(normalize=True)

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

In [7]:
baseline_pred = np.zeros(len(df))   # predict 0 for everyone
(baseline_pred == df["TARGET"]).mean()

np.float64(0.9192711805431351)

In [8]:
((baseline_pred == 1) & (df["TARGET"] == 1)).sum()

np.int64(0)

In [9]:
df.select_dtypes(include="object").columns.tolist()

/var/folders/0k/7s5007550w9b8m22j4485n740000gn/T/ipykernel_48494/3671663173.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include="object").columns.tolist()


['NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'OCCUPATION_TYPE',
 'WEEKDAY_APPR_PROCESS_START',
 'ORGANIZATION_TYPE',
 'FONDKAPREMONT_MODE',
 'HOUSETYPE_MODE',
 'WALLSMATERIAL_MODE',
 'EMERGENCYSTATE_MODE']

In [10]:
missing = df.isna().mean().sort_values(ascending=False)
missing.head(20)

COMMONAREA_MEDI             0.698723
COMMONAREA_AVG              0.698723
COMMONAREA_MODE             0.698723
NONLIVINGAPARTMENTS_MODE    0.694330
NONLIVINGAPARTMENTS_AVG     0.694330
NONLIVINGAPARTMENTS_MEDI    0.694330
FONDKAPREMONT_MODE          0.683862
LIVINGAPARTMENTS_MODE       0.683550
LIVINGAPARTMENTS_AVG        0.683550
LIVINGAPARTMENTS_MEDI       0.683550
FLOORSMIN_AVG               0.678486
FLOORSMIN_MODE              0.678486
FLOORSMIN_MEDI              0.678486
YEARS_BUILD_MEDI            0.664978
YEARS_BUILD_MODE            0.664978
YEARS_BUILD_AVG             0.664978
OWN_CAR_AGE                 0.659908
LANDAREA_MEDI               0.593767
LANDAREA_MODE               0.593767
LANDAREA_AVG                0.593767
dtype: float64

In [11]:
df.groupby(df["OWN_CAR_AGE"].isna())["TARGET"].mean()

OWN_CAR_AGE
False    0.072441
True     0.085000
Name: TARGET, dtype: float64

In [12]:
df.groupby(df["COMMONAREA_AVG"].isna())["TARGET"].mean()

COMMONAREA_AVG
False    0.069102
True     0.085742
Name: TARGET, dtype: float64

In [14]:
df[["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "DAYS_BIRTH", "DAYS_EMPLOYED"]].describe()

,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,DAYS_BIRTH,DAYS_EMPLOYED
count,3.075110e+05,3.075110e+05,307499.000000,307511.000000,307511.000000
mean,1.687979e+05,5.990260e+05,27108.573909,-16036.995067,63815.045904
std,2.371231e+05,4.024908e+05,14493.737315,4363.988632,141275.766519
min,2.565000e+04,4.500000e+04,1615.500000,-25229.000000,-17912.000000
25%,1.125000e+05,2.700000e+05,16524.000000,-19682.000000,-2760.000000
50%,1.471500e+05,5.135310e+05,24903.000000,-15750.000000,-1213.000000
75%,2.025000e+05,8.086500e+05,34596.000000,-12413.000000,-289.000000
max,1.170000e+08,4.050000e+06,258025.500000,-7489.000000,365243.000000


In [15]:
# how many applicants have the suspicious DAYS_EMPLOYED value?
(df["DAYS_EMPLOYED"] == 365243).sum()

np.int64(55374)

In [16]:
# default rate for that group vs everyone else
df.groupby(df["DAYS_EMPLOYED"] == 365243)["TARGET"].mean()

DAYS_EMPLOYED
False    0.086600
True     0.053996
Name: TARGET, dtype: float64

In [17]:
# who has the huge incomes?
df["AMT_INCOME_TOTAL"].sort_values(ascending=False).head(10)

12840     117000000.0
203693     18000090.0
246858     13500000.0
77768       9000000.0
131127      6750000.0
103006      4500000.0
204564      4500000.0
187833      4500000.0
287463      4500000.0
181698      3950059.5
Name: AMT_INCOME_TOTAL, dtype: float64

In [18]:
df_clean = df.copy()

# 1. Sentinel -> NaN, keep the information in a flag
df_clean["DAYS_EMPLOYED_ANOM"] = (df_clean["DAYS_EMPLOYED"] == 365243).astype(int)
df_clean["DAYS_EMPLOYED"] = df_clean["DAYS_EMPLOYED"].replace(365243, np.nan)

# 2. Readable age in years
df_clean["AGE_YEARS"] = -df_clean["DAYS_BIRTH"] / 365

# 3. Check the results
df_clean[["DAYS_EMPLOYED", "DAYS_EMPLOYED_ANOM", "AGE_YEARS"]].describe()

,DAYS_EMPLOYED,DAYS_EMPLOYED_ANOM,AGE_YEARS
count,252137.000000,307511.000000,307511.000000
mean,-2384.169325,0.180072,43.936973
std,2338.360162,0.384248,11.956133
min,-17912.000000,0.000000,20.517808
25%,-3175.000000,0.000000,34.008219
50%,-1648.000000,0.000000,43.150685
75%,-767.000000,0.000000,53.923288
max,0.000000,1.000000,69.120548


In [19]:
df_clean.groupby(pd.qcut(df_clean["AGE_YEARS"], 5))["TARGET"].mean()

AGE_YEARS
(20.517, 32.038]    0.112824
(32.038, 39.496]    0.093295
(39.496, 47.178]    0.078493
(47.178, 56.093]    0.067392
(56.093, 69.121]    0.051630
Name: TARGET, dtype: float64

In [20]:
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=["TARGET", "SK_ID_CURR"])
y = df_clean["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(246008, 122) (61503, 122)
0.08072908198107379 0.08072776937710356


In [21]:
# experiment: what goes wrong without stratify, on a tiny test set?
_, _, _, y_bad = train_test_split(X, y, test_size=0.001, random_state=0)
print(y_bad.mean(), len(y_bad))

0.07142857142857142 308
